## TUGAS MANDIRI 
---

### Konteks / Skenario

Manajemen platform e-commerce meminta dibuatkan **dashboard performa cabang toko** yang menggabungkan data transaksi dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi **join, window function, dan Spark SQL** —

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat **dua tabel** dan mengunggah tabel transaksi ke HDFS (tabel target cukup dibuat langsung sebagai Spark DataFrame, karena berukuran kecil dan jarang berubah — praktik umum untuk tabel referensi/*dimension table*).

In [1]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


## Instruksi Pengerjaan

Buat `SparkSession`, lalu:
1. Baca `transaksi_tugas5.csv` **dari HDFS** menjadi `df_transaksi`, tambahkan kolom `pendapatan` (`unit_terjual x harga_satuan`).
2. Buat `df_target` dari dictionary `data_target_cabang` di atas.
---

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")

SparkSession berhasil dibuat!


In [9]:
# Membaca transaksi_tugas5.csv dari HDFS menjadi df_transaksi dan menambahkan kolom pendapatan 
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv",
    header=True, inferSchema=True
)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# Membuat df_target dari dictionary data_target_cabang 
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("df_transaksi :")
df_transaksi.show(5)
print("df_target :")
df_target.show()

df_transaksi :
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows

df_target :


+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



---
**A. Join & Perbandingan Target** *(bobot 25%)*

Ringkas total `pendapatan` per `kota` dari `df_transaksi`, lalu **join** dengan `df_target`. Tambahkan kolom `pencapaian_persen`. Urutkan hasil dari pencapaian tertinggi.

In [10]:
# Meringkas total pendapatan per kota
ringkasan = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan tabel target untuk menghitung pencapaian
hasil = ringkasan.join(df_target, on="kota", how="inner")
hasil = hasil.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

hasil.orderBy(col("pencapaian_persen").desc()).show()

[Stage 8:=============================>                             (1 + 1) / 2]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



---
**B. Window Function — Kategori Terlaris per Kota** *(bobot 25%)*

Menggunakan window function, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja, gunakan `row_number()`).

In [11]:
# Kategori dengan pendapatan tertinggi di setiap kota
pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Mendefinisikan "window": ranking kategori berdasarkan total pendapatan per kota
window_func = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# Menambahkan kolom peringkat 
df_ranked = pendapatan_kategori.withColumn("rank", row_number().over(window_func))

# Menampilkan kategori dengan pendapatan tertinggi (rank 1) di setiap kota
df_ranked.filter(col("rank") <= 1).orderBy("kota").show()

+----------+--------------------+----------------+----+
|      kota|            kategori|total_pendapatan|rank|
+----------+--------------------+----------------+----+
|  Magelang|Kesehatan & Kecan...|         7275000|   1|
| Purworejo|Kesehatan & Kecan...|        10075000|   1|
|  Semarang|        Rumah Tangga|        11125000|   1|
|      Solo|Kesehatan & Kecan...|         8425000|   1|
|Yogyakarta|             Fashion|        13325000|   1|
+----------+--------------------+----------------+----+



---
**C. Spark SQL** *(bobot 25%)*

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

In [12]:
# Mendaftarkan df_transaksi dan df_target sebagai temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Menampilkan kolom kota, pic_cabang, dan jumlah transaksi di kota tersebut (diurutkan dari jumlah transaksi terbanyak)
kueri_sql = spark.sql('''
    SELECT t.kota, tg.pic_cabang, count(t.kota) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

print("Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'produk'")
print("\nKueri SQL :")
kueri_sql.show()

Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'produk'

Kueri SQL :


[Stage 16:>                                                         (0 + 2) / 2]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



---
**D. Kesimpulan** *(bobot 25%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan hasil bagian A dan B, **cabang mana yang berkinerja paling baik** dan **cabang mana yang paling perlu perhatian manajemen**? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

**Penjelasan :** Cabang yang berkinerja paling baik yaitu cabang Purworejo dengan kategori yang menghasilkan pendapatan tertinggi ada pada kategori kesehatan & kecantikan. Pencapaian cabang ini mencapai 152,167% dengan total pendapatan Rp.45.650.000, total pendapatan tersebut telah melebihi target bulanannya sebesar Rp.30.000.000 dengan selisih Rp.15.650.000. Sementara itu, cabang yang perlu perhatian manajemennya yaitu cabang Semarang dengan kategori yang menghasilkan pendapatan tertinggi ada pada kategori rumah tangga. Pencapaian cabang ini hanya mencapai 69,409% dengan total pendapatan Rp.38.175.000, sedangkan target bulanannya sebesar Rp.55.000.000. Oleh karenanya, pendapatan pada cabang Semarang masih memiliki selisih sebesar 16.825.000 untuk mencapai target yang ditentukan. Cabang ini perlu mendapat perhatian manajemen untuk meningkatkan pendapatan agar capaiannya dapat memenuhi target bulanan yang telah ditentukan.

---

## Menutup SparkSession

Kebiasaan baik: selalu tutup SparkSession di akhir notebook untuk clear up resources (memori & CPU) yang dipakainya.

In [13]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
